# Mini projekt PAD - scrapper danych o oponach
## Jakub Michalak S20034
### Opis projektu
- Projekt polega na pobraniu danych o oponach z dwóch sklepów internetowych: sklep opon i oponeo
- Dane pobrane z obu sklepów zostaną zapisane w plikach CSV
- Dane zostaną oczyszczone i przygotowane do analizy

In [2]:
import pandas as pd
import re
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

# Scrapping danych
### Konfiguracja drivera do scrapowania danych 
- **Uwaga**: testowane na szerokości okna 1110 pikseli
- Ustawienie szerokości okna na 1100 pikseli pozwala na poprawne działanie skryptów do scrapowania danych ze stron sklep opon i oponeo. Przy wyższej szerokości okna mogą wystąpić problemy z lokalizacją elementów na stronie (np. oceny).

In [117]:
download_service = Service()
driver = webdriver.Chrome(service=download_service)
driver.set_window_size(1100, 800)

sklep_opon_base_url = "https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs="
oponeo_base_url = "https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16"

### Funkcje obsługujące scrapowanie danych ze stron sklep opon i oponeo

In [3]:
def close_sklep_opon_popups(outer_driver):
    try:
        btn_cookie = outer_driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
        btn_cookie.click()
        print("Przycisk akceptacji ciasteczek został kliknięty.")
    except NoSuchElementException:
        print("Przycisk akceptacji ciasteczek nie został znaleziony.")
    except Exception as exception:
        print("Nie udało się kliknąć przycisku akceptacji ciasteczek:", exception)
    
    try:
        outer_driver.execute_script("""
            const shadowHost = document.querySelector("body > div.gr-visual-prompt");
            if (shadowHost) {
                const shadowRoot = shadowHost.shadowRoot;
                const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
                if (closeButton) {
                    closeButton.click();
                    console.log("Okienko powiadomień zostało zamknięte.");
                } else {
                    console.log("Nie znaleziono przycisku zamknięcia powiadomień.");
                }
            } else {
                console.log("Okno powiadomień nie zostało znalezione.");
            }
        """)
    except Exception as exception:
        print("Nie udało się zamknąć okienka powiadomień:", exception)

def close_oponeo_popup(outer_driver):
    try:
        reject_button = outer_driver.find_element(By.CSS_SELECTOR, "#consentsBar > div.buttonsContainer.container > div > span.reject")
        reject_button.click()
        print("Okienko prywatności zostało zamknięte.")
    except NoSuchElementException:
        print("Okienko prywatności nie jest widoczne lub zostało już zamknięte.")
    except Exception as e:
        print("Wystąpił błąd podczas zamykania okienka prywatności:", e)
        
def load_sklep_opon_tire_data(outer_driver):
    scrapped_data = []
    try:
        opony_elements = outer_driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
            
        class_mapping = {
            "Premium": "Premium",
            "Średnia": "Średnia",
            "Średniej": "Średnia",
            "Ekonomiczna": "Ekonomiczna",
            "Ekonomicznej": "Ekonomiczna"
        }
        
        for opona_element in opony_elements:
            
            try:
                load_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="li"]').text
                speed_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="si"]').text
            except NoSuchElementException:
                load_index = None
                speed_index = None
            
            noise_level = None
            try:
                noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
                for noise_level_element in noise_level_elements:
                    text = noise_level_element.text
                    match = re.search(r'\d+', text)
                    if match:
                        noise_level = int(match.group())
            except (NoSuchElementException, IndexError):
                pass
            
            etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
            fuel_index = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
            wet_grip_index = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
            noise_index = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
            
            try:
                tire_class_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
                tire_class_text = tire_class_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
                tire_class = class_mapping.get(tire_class_text, tire_class_text)
            except NoSuchElementException:
                tire_class = None
                
            try:
                user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
                user_rating = float(user_rating_element.text.replace(",", "."))
            except NoSuchElementException:
                user_rating = None
                
            price = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
            
            # Pobranie dostępności
            try:
                availability_element = opona_element.find_element(By.CSS_SELECTOR, "div.tooltip-product-listing")
                availability_code = availability_element.get_attribute("data-attribute-code")
                
                if availability_code == "item_availability_tooltip_high":
                    availability = "full"
                elif availability_code == "item_availability_tooltip_medium":
                    availability = "medium"
                elif availability_code == "item_availability_tooltip_low":
                    availability = "low"
                elif availability_code == "item_availability_tooltip_last":
                    availability = "last"
                else:
                    availability = None
            except NoSuchElementException:
                availability = None
                
            tire_data = {
                "name": opona_element.get_attribute("data-ee-product-properties").split(";")[0].split(":")[1],
                "brand": opona_element.get_attribute("data-ee-product-properties").split(";")[4].split(":")[1],
                "model": opona_element.get_attribute("data-ee-product-properties").split(";")[6].split(":")[1],
                "size": opona_element.get_attribute("data-ee-product-properties").split(";")[5].split(":")[1],
                "load_index": load_index,
                "speed_index": speed_index,
                "fuel_index": fuel_index,
                "wet_grip_index": wet_grip_index,
                "noise_index": noise_index,
                "noise_level": noise_level,
                "class": tire_class,
                "user_rating": user_rating,
                "price": price,
                "availability": availability
            }
            
            scrapped_data.append(tire_data)
    finally:
        pass
    return scrapped_data   

def load_next_oponeo_page(web_driver, current_page_number):
    try:
        next_page_button = web_driver.find_element(By.ID, f"_ctPgrp_pi{current_page_number}i")
        next_page_button.click()
        time.sleep(2)
        return True
    except NoSuchElementException:
        return False
    
def load_oponeo_tire_data(outer_driver):
    scrapped_data = []
    products = outer_driver.find_elements(By.CLASS_NAME, "product")
    
    for product in products:
        try:
            try:
                link_element = product.find_element(By.CSS_SELECTOR, ".productName a")
                nazwa = link_element.get_attribute("title")
            except NoSuchElementException:
                nazwa = product.find_element(By.CLASS_NAME, "productName").text

            noise = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text
            match = re.search(r'\d+', noise)
            if match:
                noise_level = int(match.group())
            else:
                noise_level = int(product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[1].replace("dB", "").strip())
             
            noise_index = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[0] 
            noise_index_text = noise_index if noise_index in {"A", "B", "C", "D", "E", "F"} and len(noise_index) == 1 else None   
                
            try:
                user_rating = product.find_element(By.CSS_SELECTOR, ".productRating .note").text
                user_rating = user_rating.replace(',', '.')
            except NoSuchElementException:
                user_rating = None
                
            # Pobranie poziomu dostępności
            try:
                stock_level_element = product.find_element(By.CSS_SELECTOR, ".stockLevel")
                stock_level_class = stock_level_element.get_attribute("class").split()[-1]
                
                if stock_level_class == "full":
                    availability = "full"
                elif stock_level_class == "medium":
                    availability = "medium"
                elif stock_level_class == "low":
                    availability = "low"
                else:
                    availability = None
            except NoSuchElementException:
                availability = None
                
            tire_info = {
                "name": nazwa,
                "brand": product.find_element(By.CLASS_NAME, "producerName").text,
                "model": product.find_element(By.CLASS_NAME, "modelName").text,
                "size": product.find_element(By.CLASS_NAME, "modelSize").text,
                "load_index": product.find_element(By.XPATH, ".//span[@data-tp='TireLoadIndex']/em").text,
                "speed_index": product.find_element(By.XPATH, ".//span[@data-tp='TireSpeedIndex']/em").text,
                "fuel_index": product.find_element(By.CSS_SELECTOR, ".icon-fuel em").text,
                "wet_grip_index": product.find_element(By.CSS_SELECTOR, ".icon-rain em").text,
                "noise_index": noise_index_text,
                "noise_level": noise_level,
                "class": product.find_element(By.CLASS_NAME, "class").text.replace("KLASA ", "").capitalize(),
                "user_rating": user_rating,
                "price": product.find_element(By.CLASS_NAME, "priceValue").text,
                "availability": availability
            }
            
            scrapped_data.append(tire_info)

        except NoSuchElementException:
            pass
            
    return scrapped_data   

def update_column_types():
    # Konwersja kolumn do typów numerycznych w zbiorze df_oponeo
    df_oponeo['noise_level'] = pd.to_numeric(df_oponeo['noise_level'], errors='coerce').astype('Int64')
    df_oponeo['user_rating'] = pd.to_numeric(df_oponeo['user_rating'], errors='coerce').astype(float)
    df_oponeo['price'] = pd.to_numeric(df_oponeo['price'], errors='coerce').astype(float)
    
    # Konwersja kolumn do typów numerycznych w zbiorze df_sklep_opon
    df_sklep_opon['noise_level'] = pd.to_numeric(df_sklep_opon['noise_level'], errors='coerce').astype('Int64')
    df_sklep_opon['user_rating'] = pd.to_numeric(df_sklep_opon['user_rating'], errors='coerce').astype(float)
    df_sklep_opon['price'] = pd.to_numeric(df_sklep_opon['price'], errors='coerce').astype(float)
    
    # Konwersja kolumn z ograniczoną liczbą unikalnych wartości na typ 'category' w zbiorze df_sklep_opon
    df_sklep_opon['size'] = df_sklep_opon['size'].astype('category')
    df_sklep_opon['load_index'] = df_sklep_opon['load_index'].astype('category')
    df_sklep_opon['speed_index'] = df_sklep_opon['speed_index'].astype('category')
    df_sklep_opon['fuel_index'] = df_sklep_opon['fuel_index'].astype('category')
    df_sklep_opon['wet_grip_index'] = df_sklep_opon['wet_grip_index'].astype('category')
    df_sklep_opon['noise_index'] = df_sklep_opon['noise_index'].astype('category')
    df_sklep_opon['class'] = df_sklep_opon['class'].astype('category')
    df_sklep_opon['availability'] = df_sklep_opon['availability'].astype('category')
    
    # Konwersja kolumn z ograniczoną liczbą unikalnych wartości na typ 'category' w zbiorze df_oponeo
    df_oponeo['size'] = df_oponeo['size'].astype('category')
    df_oponeo['load_index'] = df_oponeo['load_index'].astype('category')
    df_oponeo['speed_index'] = df_oponeo['speed_index'].astype('category')
    df_oponeo['fuel_index'] = df_oponeo['fuel_index'].astype('category')
    df_oponeo['wet_grip_index'] = df_oponeo['wet_grip_index'].astype('category')
    df_oponeo['noise_index'] = df_oponeo['noise_index'].astype('category')
    df_oponeo['class'] = df_oponeo['class'].astype('category')
    df_oponeo['availability'] = df_oponeo['availability'].astype('category')
    
    # Optymalizacja pamięci - zmiana typów numerycznych na bardziej optymalne
    df_oponeo['user_rating'] = df_oponeo['user_rating'].astype('float32')
    df_oponeo['price'] = df_oponeo['price'].astype('float32')
    
    # Optymalizacja pamięci - zmiana typów numerycznych na bardziej optymalne
    df_sklep_opon['user_rating'] = df_sklep_opon['user_rating'].astype('float32')
    df_sklep_opon['price'] = df_sklep_opon['price'].astype('float32')

### Kod do pobierania danych ze strony sklep opon

In [118]:
sklep_opon_tires_data = []
offset = 0

while True:
    url = f"{sklep_opon_base_url}{offset}"
    driver.get(url)
    time.sleep(4)
    
    if offset == 0:
        close_sklep_opon_popups(driver)
    
    tires_data = load_sklep_opon_tire_data(driver)
    sklep_opon_tires_data.extend(tires_data)
    
    offset += 20
    if len(driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')) == 0:
        print("Brak nowych danych. Koniec paginacji.")
        break

df_sklep_opon = pd.DataFrame(sklep_opon_tires_data)
display(df_sklep_opon)

Przycisk akceptacji ciasteczek został kliknięty.
Brak nowych danych. Koniec paginacji.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price,availability
0,Wintrac 205/55 R16 91 H,Vredestein,Wintrac,205/55 R16,91,H,C,B,B,70.0,Premium,5.4,376.99,full
1,Winguard Snow'G WH2 205/55 R16 91 H,Nexen,Winguard Snow'G WH2,205/55 R16,91,H,D,C,B,70.0,Średnia,5.3,310.00,full
2,DIMAX ALPINE 205/55 R16 94 H,Radar,DIMAX ALPINE,205/55 R16,94,H,D,C,A,69.0,Ekonomiczna,5.2,225.49,full
3,Frigo HP2 205/55 R16 91 H,Dębica,Frigo HP2,205/55 R16,91,H,C,C,B,72.0,Ekonomiczna,5.2,299.00,full
4,Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71.0,Ekonomiczna,5.1,239.00,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,Winter Sottozero Serie II 205/55 R16 94 V,Pirelli,Winter Sottozero Serie II,205/55 R16,94,V,C,C,B,72.0,Premium,5.5,1059.57,last
402,WINTERPRO2 (EVO)* 205/55 R16 91 T,Gt radial,WINTERPRO2 (EVO)*,205/55 R16,91,T,D,B,B,70.0,None,NaN,340.47,last
403,Ultra Grip 8 205/55 R16 91 T,Goodyear,Ultra Grip 8,205/55 R16,91,T,D,D,B,71.0,Premium,5.2,374.43,last
404,Blizzak LM005 205/55 R16 94 V,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71.0,Premium,5.5,643.11,last


### Kod do pobierania danych ze strony oponeo

In [111]:
driver.get(oponeo_base_url)
close_oponeo_popup(driver)

all_tires_data = []

page_number = 1
while True:
    tires_data = load_oponeo_tire_data(driver)
    all_tires_data.extend(tires_data)
    
    page_number += 1
    if not load_next_oponeo_page(driver, page_number):
        print("Brak nowych danych. Koniec paginacji.")
        break

df_oponeo = pd.DataFrame(all_tires_data)
display(df_oponeo)

Okienko prywatności zostało zamknięte.
Brak nowych danych. Koniec paginacji.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price,availability
0,Bridgestone Blizzak LM005 205/55 R16 91 H,Bridgestone,Blizzak LM005,205/55 R16,91,H,C,A,B,71,Premium,4.7,459,full
1,Michelin Alpin 7 205/55 R16 91 H,Michelin,Alpin 7,205/55 R16,91,H,C,B,B,71,Premium,4.8,467,full
2,Dębica Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71,Ekonomiczna,4.2,252,full
3,Kormoran Snow 205/55 R16 91 H,Kormoran,Snow,205/55 R16,91,H,D,C,B,72,Ekonomiczna,4.5,249,full
4,Firemax FM805+ 205/55 R16 91 H,Firemax,FM805+,205/55 R16,91,H,D,C,A,67,Ekonomiczna,4.4,209,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,Continental ContiWinterContact TS830 P 205/55 ...,Continental,ContiWinterContact TS830 P,205/55 R16,91,H,D,C,B,72,Premium,4.4,760,medium
218,Goodyear UG Performance 2 205/55 R16 91 H RUN ...,Goodyear,UG Performance 2,205/55 R16,91,H,D,C,B,72,Premium,4.3,778,full
219,Bridgestone Blizzak LM005 205/55 R16 94 V XL,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71,Premium,4.7,854,medium
220,Pirelli SottoZero Serie 3 205/55 R16 91 H RUN ...,Pirelli,SottoZero Serie 3,205/55 R16,91,H,D,B,B,72,Premium,4.6,981,full


### Zamknięcie drivera po scrapowaniu danych

In [121]:
driver.quit()

### Zapisanie danych do plików CSV
- służy to do zapisania danych do plików CSV, aby można było je łatwo odtworzyć w przyszłości
- dane zapisane w plikach CSV można wczytać do DataFrame za pomocą funkcji `pd.read_csv`

In [3]:
# df_sklep_opon.to_csv("data/miniprojekt/sklep_opon.csv", index=False)
# df_oponeo.to_csv("data/miniprojekt/oponeo.csv", index=False)
df_sklep_opon = pd.read_csv("data/miniprojekt/sklep_opon.csv", delimiter=",")
df_oponeo = pd.read_csv("data/miniprojekt/oponeo.csv", delimiter=",")

### Weryfikacja poprawnych wartości w każdej kolumnie

In [10]:
print("sklep opon")
print("typy danych w kolumnach")
print(df_sklep_opon.dtypes)
print("unikalne wartości w kolumnach")
for column in df_sklep_opon.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_sklep_opon[column].unique())
print("oponeo")
print("typy danych w kolumnach")
print(df_oponeo.dtypes)
print("unikalne wartości w kolumnach")
for column in df_oponeo.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_oponeo[column].unique())

sklep opon
typy danych w kolumnach
name               object
brand              object
model              object
size               object
load_index          int64
speed_index        object
fuel_index         object
wet_grip_index     object
noise_index        object
noise_level         Int64
class              object
user_rating       float64
price             float64
availability       object
dtype: object
unikalne wartości w kolumnach
size ['205/55 R16']
load_index [94 91]
speed_index ['H' 'T' 'V' 'R']
fuel_index [nan 'D' 'C' 'E' 'B']
wet_grip_index [nan 'B' 'A' 'C' 'E' 'D']
noise_index [nan 'A' 'B' 'C']
noise_level <IntegerArray>
[<NA>, 68, 69, 71, 70, 72, 67]
Length: 7, dtype: Int64
class ['Ekonomiczna' 'Średnia' 'Premium']
user_rating [nan 5.3 5.5 5.7 5.4 5.  4.6 5.2 5.6 4.3 5.1 4.5 6.  4.8 4.4 4.9 4.7 4.2
 1.  2.5 2.3 1.4]
price [ 255.92  280.57  281.68  412.66  728.52 1168.49  826.82  460.    551.68
  306.31  527.16  536.31  444.94  502.64  619.55  587.41  496.3   450.99
  469

# Czyszczenie i przygotowanie danych

### #0. konwersja kolumn do typów numerycznych i typu category

In [18]:
update_column_types()

### Funkcje pomocnicze

In [6]:
def find_closest_key(value, param_map):
        closest_key = min(param_map, key=lambda k: abs(param_map[k] - value))
        return closest_key

def find_closest_value(value, param_map):
        closest_key = min(param_map, key=lambda k: abs(param_map[k] - value))
        return param_map[closest_key]

### #1. Uzupełnianie noise_index na podstawie noise_level

In [7]:
def update_noise_index_column(df):
    grouped_noise_index = df.groupby('noise_index')['noise_level'].mean().round().astype(int)
    
    if 'C' not in grouped_noise_index:
        if 'B' in grouped_noise_index and 'A' in grouped_noise_index:
            grouped_noise_index['C'] = grouped_noise_index['B'] + (grouped_noise_index['B'] - grouped_noise_index['A'])
    
    noise_map = grouped_noise_index.to_dict()
    
    def apply_update_noise_index(row):
        if pd.notna(row['noise_level']) and pd.isna(row['noise_index']):
            return find_closest_key(row['noise_level'], noise_map)
        return row['noise_index']
    
    df['noise_index'] = df.apply(apply_update_noise_index, axis=1)
    
    return df

df_oponeo = update_noise_index_column(df_oponeo)
df_sklep_opon = update_noise_index_column(df_sklep_opon)

print("Zaktualizowano noise_index")
print(df_oponeo['noise_index'].unique())
print(df_sklep_opon['noise_index'].unique())

Zaktualizowano noise_index
['B' 'A' 'C']
['B' 'A' nan 'C']


### #2. Uzupełnianie ceny na podstawie średniej ceny i klasy na podstawie ceny

In [8]:
def update_price_column(df):
    class_price_map = df.groupby('class')['price'].mean().to_dict()
    
    def apply_update_price(row):
        if pd.isna(row['price']):
            return find_closest_value(row['price'], class_price_map)
        return row['price']
    
    def apply_update_class(row):
        if pd.isna(row['class']):
            return find_closest_key(row['price'], class_price_map)
        return row['class']
    
    df['price'] = df.apply(apply_update_price, axis=1)
    df['class'] = df.apply(apply_update_class, axis=1)
    
    return df

df_oponeo = update_price_column(df_oponeo)
df_sklep_opon = update_price_column(df_sklep_opon)

print("Zaktualizowano cenę i klasę")
print("oponeo")
print(df_oponeo['price'].unique())
print(df_oponeo['class'].unique())
print("sklep opon")
print(df_sklep_opon['price'].unique())
print(df_sklep_opon['class'].unique())

Zaktualizowano cenę i klasę
oponeo
[459.         467.         252.         249.         209.
 191.         207.         215.         216.         217.
 218.         220.         229.         254.         257.
 264.         269.         293.         297.         299.
 307.         332.         337.         338.         339.
 340.         342.         346.         348.         353.
 357.         373.         379.         382.         383.
 387.         388.         390.         392.         393.
 407.         409.         424.         432.         447.
 452.         466.         468.         474.         477.
 520.         555.         597.         258.         267.
 270.         271.         273.         274.         275.
 279.         283.         287.         292.         294.
 298.         300.         301.         304.         305.
 311.         315.         317.         320.         325.
 330.         331.         333.         334.         335.
 336.         345.         350.      

### #3. Znalezienie, uzupełnienie i usunięcie duplikatów
- Funkcja do identyfikacji duplikatów na podstawie 'name', 'brand', 'model' i wypełniania brakujących danych
- Usuwa duplikaty na podstawie ilości uzupełnionych kolumn - zostawia wiersz z największą ilością uzupełnionych kolumn

In [9]:
def fill_missing_with_duplicates(df_param):
    # Krok 1: Znalezienie duplikatów
    duplicates = df_param[df_param.duplicated(subset=['name', 'brand', 'model'], keep=False)]
    
    # Krok 2: Iteracja po grupach duplikatów
    for idx, group in duplicates.groupby(['name', 'brand', 'model']):
        for col in df_param.columns:
            if group[col].isna().any():
                # Jeśli mamy wartości do wypełnienia, to szukamy pierwszej dostępnej wartości
                fill_value = group[col].dropna().iloc[0] if not group[col].dropna().empty else None
                if fill_value is not None:  # Tylko wypełniaj, jeśli fill_value nie jest None
                    df_param.loc[group.index, col] = df_param.loc[group.index, col].fillna(fill_value)

    # Krok 3: Dodanie kolumny pomocniczej 'non_nan_count', która liczy liczbę nie-NaN wierszy
    df_param['non_nan_count'] = df_param.notna().sum(axis=1)
    
    # Krok 4: Grupowanie po duplikatach i wybieranie wiersza z największą liczbą nie-NaN
    df_param = df_param.loc[df_param.groupby(['name', 'brand', 'model'])['non_nan_count'].idxmax()]

    # Krok 5: Usunięcie kolumny pomocniczej
    df_param.drop(columns=['non_nan_count'], inplace=True)
    
    return df_param

print("Przed usunięciem duplikatów")
print(df_oponeo.shape)
print(df_sklep_opon.shape)

df_oponeo = fill_missing_with_duplicates(df_oponeo)
df_sklep_opon = fill_missing_with_duplicates(df_sklep_opon)

print("Po usunięciu duplikatów")
print(df_oponeo.shape)
print(df_sklep_opon.shape)

Przed usunięciem duplikatów
(222, 14)
(406, 14)
Po usunięciu duplikatów
(208, 14)
(289, 14)


### #4. Usuwanie wierszy z brakującymi wartościami w istotnych kolumnach

In [11]:
df_oponeo.dropna(subset=['fuel_index', 'wet_grip_index', 'noise_index'], inplace=True)
df_sklep_opon.dropna(subset=['fuel_index', 'wet_grip_index', 'noise_index'], inplace=True)

df_oponeo.dropna(subset=['availability'], inplace=True)
df_sklep_opon.dropna(subset=['availability'], inplace=True)

print("Po usunięciu braków danych")
print(df_oponeo.shape)
print(df_sklep_opon.shape)

Po usunięciu braków danych
(203, 14)
(219, 14)


### #5. Zastąpienie brakujących wartości w user_rating na 0

In [13]:
df_oponeo['user_rating'] = df_oponeo['user_rating'].fillna(0)
df_sklep_opon['user_rating'] = df_sklep_opon['user_rating'].fillna(0)

### #6. Wyświetlenie danych po czyszczeniu i zapisanie do plików CSV

In [24]:
print("sklep opon")
print("typy danych w kolumnach")
print(df_sklep_opon.dtypes)
print("unikalne wartości w kolumnach")
for column in df_sklep_opon.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_sklep_opon[column].unique())
display(df_sklep_opon)

print("oponeo")
print("typy danych w kolumnach")
print(df_oponeo.dtypes)
print("unikalne wartości w kolumnach")
for column in df_oponeo.columns:
    if column not in ["name", "brand", "model"]:
        print(column, df_oponeo[column].unique())
display(df_oponeo)

df_oponeo.to_csv("data/miniprojekt/oponeo_cleaned.csv", index=False)
df_sklep_opon.to_csv("data/miniprojekt/sklep_opon_cleaned.csv", index=False)

sklep opon
typy danych w kolumnach
name                object
brand               object
model               object
size              category
load_index        category
speed_index       category
fuel_index        category
wet_grip_index    category
noise_index       category
noise_level          Int64
class             category
user_rating        float32
price              float32
availability      category
dtype: object
unikalne wartości w kolumnach
size ['205/55 R16']
Categories (1, object): ['205/55 R16']
load_index [91, 94]
Categories (2, int64): [91, 94]
speed_index ['H', 'T', 'V', 'R']
Categories (4, object): ['H', 'R', 'T', 'V']
fuel_index ['D', 'C', 'E', 'B']
Categories (4, object): ['B', 'C', 'D', 'E']
wet_grip_index ['B', 'A', 'C', 'E', 'D']
Categories (5, object): ['A', 'B', 'C', 'D', 'E']
noise_index ['A', 'B', 'C']
Categories (3, object): ['A', 'B', 'C']
noise_level <IntegerArray>
[68, 69, 71, 70, 72, 67]
Length: 6, dtype: Int64
class ['Premium', 'Średnia', 'Ekonomiczna'

,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price,availability
0,Alpin 5 205/55 R16 91 H,Michelin,Alpin 5,205/55 R16,91,H,D,B,A,68,Premium,5.5,728.520020,full
1,Alpin 6 205/55 R16 91 H,Michelin,Alpin 6,205/55 R16,91,H,C,B,A,69,Premium,5.7,1168.489990,medium
2,Alpin 6 205/55 R16 91 T,Michelin,Alpin 6,205/55 R16,91,T,C,B,A,69,Premium,5.7,826.820007,full
3,Alpin 7 205/55 R16 91 H,Michelin,Alpin 7,205/55 R16,91,H,C,B,B,71,Premium,5.4,460.000000,full
4,Alpin 7 205/55 R16 91 T,Michelin,Alpin 7,205/55 R16,91,T,C,B,B,71,Premium,5.4,460.000000,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,Z507 205/55 R16 91 V,Goodride,Z507,205/55 R16,91,V,C,C,B,72,Ekonomiczna,5.1,227.000000,full
215,g-Force Winter 2 205/55 R16 91 H,Bfgoodrich,g-Force Winter 2,205/55 R16,91,H,D,B,A,69,Średnia,5.2,464.989990,full
216,g-Force Winter 2 205/55 R16 91 T,Bfgoodrich,g-Force Winter 2,205/55 R16,91,T,D,B,A,69,Średnia,5.2,559.059998,full
217,g-Force Winter 2 205/55 R16 94 H,Bfgoodrich,g-Force Winter 2,205/55 R16,94,H,D,B,A,69,Średnia,5.2,576.359985,medium


# Eksploracyjna Analiza danych

- Opcjonalne ładowanie oczyszczonych danych z plików CSV jeżeli nie chcemy ponownie scrapować danych

In [7]:
df_oponeo = pd.read_csv("data/miniprojekt/oponeo_cleaned.csv", delimiter=",")
df_sklep_opon = pd.read_csv("data/miniprojekt/sklep_opon_cleaned.csv", delimiter=",")
update_column_types()

df_oponeo['retailer'] = 'Oponeo'
df_sklep_opon['retailer'] = 'Sklep opon'
df = pd.concat([df_oponeo, df_sklep_opon], ignore_index=True)

### #1. Średnia, mediana, min i max cen opon w każdym sklepie

In [8]:
price_stats = df.groupby('retailer')['price'].agg(['mean', 'median', 'min', 'max'])
price_stats

,mean,median,min,max
retailer,,,,
Oponeo,426.227905,400.000000,191.000000,854.00000
Sklep opon,480.524902,419.570007,197.089996,3542.51001


## #2. Średnia cena opon w zależności od marki i sklepu

In [9]:
brand_price_stats = df.groupby(['retailer', 'brand'])['price'].mean().unstack()
brand_price_stats

brand,Aplus,Apollo,Arivo,Austone,Autogreen,BFGoodrich,Barum,Bfgoodrich,Bridgestone,Ceat,...,Triangle,Tristar,Uniroyal,Viking,Vredestein,West Lake,Windforce,Yokohama,ZMax,Zeetex
retailer,,,,,,,,,,,,,,,,,,,,,
Oponeo,320.0,376.333344,267.000000,380.000000,NaN,582.0,519.400024,NaN,576.090881,265.50000,...,279.000000,370.0,443.666656,479.000000,445.166656,341.0,421.000000,478.75000,NaN,264.0
Sklep opon,NaN,NaN,292.855011,303.584991,321.720001,NaN,484.019989,645.472473,566.739990,400.73999,...,221.580002,NaN,415.672485,553.960022,424.725983,NaN,383.787506,387.14502,360.809998,NaN


### #3. Średnia cena opon w zależności od klasy i sklepu

In [13]:
class_price_stats = df.groupby(['retailer', 'class'], observed=False)['price'].mean().unstack()
class_price_stats

class,Ekonomiczna,Premium,Średnia
retailer,,,
Oponeo,358.990112,536.123230,433.372101
Sklep opon,324.129700,655.906372,434.269318


### #4. Średnia ocena użytkownika w każdym sklepie

In [15]:
rating_stats = df.groupby('retailer')['user_rating'].mean()
rating_stats

retailer
Oponeo        3.966010
Sklep opon    4.489041
Name: user_rating, dtype: float32

### #5. Liczba opon w każdej klasie efektywności w obu sklepach

In [17]:
class_distribution = df.groupby(['retailer', 'class'], observed=False).size().unstack()
class_distribution

class,Ekonomiczna,Premium,Średnia
retailer,,,
Oponeo,101,59,43
Sklep opon,67,79,73


### #6. Średnia ceny opon w zależności od:
 - sklepu oraz indeksów: paliwowego, przyczepności i hałasu
 - sklepu oraz nośności
 - sklepu oraz indeksu prędkości

In [63]:
pivot_price_indices = df.pivot_table(
    values='price',
    index='retailer',
    columns=['fuel_index', 'wet_grip_index', 'noise_index'],
    aggfunc='mean',
    observed=False
)
display(pivot_price_indices)

pivot_price_speed_load = df.pivot_table(
    values='price',
    index='retailer',
    columns='speed_index',
    aggfunc='mean',
    observed=False
)
display(pivot_price_speed_load)

pivot_price_load = df.pivot_table(
    values='price',
    index='retailer',
    columns='load_index',
    aggfunc='mean',
    observed=False
)
display(pivot_price_load)

fuel_index          B                                   C              \
wet_grip_index      B           C           D           A           B   
noise_index         A           B           A           B           A   
retailer                                                                
Oponeo          577.0  427.000000         NaN  623.000000  502.875000   
Sklep opon      435.0  389.221436  634.390015  601.054016  569.471558   

fuel_index                                                   ...           D  \
wet_grip_index                  C                         D  ...           C   
noise_index              B      A           B      C      A  ...           C   
retailer                                                     ...               
Oponeo          465.239990  379.0  381.805542  300.0  283.0  ...         NaN   
Sklep opon      473.080536    NaN  389.586182    NaN    NaN  ...  807.344971   

fuel_index                                      E                         \
wet_grip_index          D           E           B                      C   
noise_index             B           B           A          B           B   
retailer                                                                   
Oponeo          309.37500         NaN  359.268524  421.00000  410.571442   
Sklep opon      812.08252  358.859985         NaN  378.63501  448.428345   

fuel_index                                         F  
wet_grip_index           D           E      F      C  
noise_index              B           B      B      B  
retailer                                              
Oponeo          405.333344  393.000000  423.0  363.5  
Sklep opon             NaN  581.780029    NaN    NaN  

[2 rows x 27 columns]

speed_index,H,R,T,V
retailer,,,,
Oponeo,425.361481,NaN,421.410248,434.527771
Sklep opon,439.631470,634.390015,531.429199,546.580505


load_index,91,94
retailer,,
Oponeo,412.209076,457.380951
Sklep opon,465.595795,508.056519


### #7. Porównanie:
 - cen opon w obu sklepach dla tych samych modeli
  - dostępności opon w obu sklepach dla tych samych modeli

In [64]:
unique_df_price = df.groupby(['brand', 'model', 'retailer'], as_index=False)['price'].min()

df_oponeo = unique_df_price[unique_df_price['retailer'] == 'Oponeo'][['brand', 'model', 'price']].rename(columns={'price': 'Oponeo'})
df_sklep_opon = unique_df_price[unique_df_price['retailer'] == 'Sklep opon'][['brand', 'model', 'price']].rename(columns={'price': 'Sklep opon'})

price_comparison_df = pd.merge(df_oponeo, df_sklep_opon, on=['brand', 'model'], how='inner')

price_comparison_df['cheaper_in'] = price_comparison_df.apply(
    lambda row: 'Oponeo' if row['Oponeo'] < row['Sklep opon'] else 'Sklep opon' if row['Oponeo'] > row['Sklep opon'] else 'Same price',
    axis=1
)

display(price_comparison_df)

unique_df_availability = df.groupby(['brand', 'model', 'retailer'], as_index=False)['availability'].max()

df_oponeo = unique_df_availability[unique_df_availability['retailer'] == 'Oponeo'][['brand', 'model', 'availability']].rename(columns={'availability': 'Oponeo'})
df_sklep_opon = unique_df_availability[unique_df_availability['retailer'] == 'Sklep opon'][['brand', 'model', 'availability']].rename(columns={'availability': 'Sklep opon'})

availability_comparison_df = pd.merge(df_oponeo, df_sklep_opon, on=['brand', 'model'], how='inner')

availability_comparison_df['more_available_in'] = availability_comparison_df.apply(
    lambda row: 'Oponeo' if row['Oponeo'] > row['Sklep opon'] else 'Sklep opon' if row['Oponeo'] < row['Sklep opon'] else 'Taka sama dostępność',
    axis=1
)

display(availability_comparison_df)

,brand,model,Oponeo,Sklep opon,cheaper_in
0,Austone,SP901,380.0,224.990005,Sklep opon
1,Barum,Polaris 5,655.0,534.429993,Sklep opon
2,Barum,Polaris 6,367.0,289.000000,Sklep opon
3,Bridgestone,Blizzak 6,468.0,444.940002,Sklep opon
4,Bridgestone,Blizzak LM001,465.0,587.409973,Oponeo
5,Bridgestone,Blizzak LM001 EVO,491.0,496.299988,Oponeo
6,Bridgestone,Blizzak LM005,459.0,450.989990,Sklep opon
7,Bridgestone,Blizzak LM005 DriveGuard,631.0,653.929993,Oponeo
8,Continental,WinterContact TS 860 S,582.0,595.309998,Oponeo
9,Continental,WinterContact TS 870,464.0,460.000000,Sklep opon


,brand,model,Oponeo,Sklep opon,more_available_in
0,Austone,SP901,full,medium,Sklep opon
1,Barum,Polaris 5,full,medium,Sklep opon
2,Barum,Polaris 6,medium,full,Oponeo
3,Bridgestone,Blizzak 6,medium,medium,Taka sama dostępność
4,Bridgestone,Blizzak LM001,full,low,Sklep opon
5,Bridgestone,Blizzak LM001 EVO,full,full,Taka sama dostępność
6,Bridgestone,Blizzak LM005,medium,medium,Taka sama dostępność
7,Bridgestone,Blizzak LM005 DriveGuard,medium,full,Oponeo
8,Continental,WinterContact TS 860 S,full,full,Taka sama dostępność
9,Continental,WinterContact TS 870,medium,medium,Taka sama dostępność
